# PWV03_longscale : Two-Point Temporal Correlation Function — long timescales (> night)

## Overview

This notebook extends the two-point temporal correlation analysis of
`PWV03_TwoPoint_TemporalCorrelation_separateFilters.ipynb` to **multi-day
timescales**, probing whether the PWV at Cerro Pachon is correlated
across successive nights and over periods of several days to weeks.

## Key difference from PWV03 (intra-night)

In PWV03 the nightly mean is subtracted before forming pairs, and only
same-night pairs are used.  This is optimal for measuring the
*intra-night* decorrelation (minutes to hours).  Here we want to probe
*inter-night and multi-day* correlations, so two changes are necessary:

1. **No nightly mean subtraction**: subtracting per-night means would remove
   exactly the signal we want to measure (the slow PWV modulation driven by
   synoptic weather and the seasonal cycle).  Instead we subtract the
   **global mean** over the full dataset, so that inter-night offsets
   are preserved in the residual $\delta\mathrm{PWV}$.

2. **All pairs included** (same-night and cross-night): the pair formation
   is not restricted to a single night, allowing lags from 1 minute to
   the full multi-year span of the dataset.

## Statistical estimator

$$
C(\Delta t) =
  \frac{\bigl\langle \delta\mathrm{PWV}(t_i)\,\delta\mathrm{PWV}(t_j)\bigr\rangle_{\Delta t_{ij}\in\mathrm{bin}}}
       {\sigma^2_{\mathrm{PWV}}}
\qquad
\delta\mathrm{PWV}(t) = \mathrm{PWV}(t) - \langle\mathrm{PWV}\rangle_{\mathrm{global}}
$$

The variance $\sigma^2$ is the global variance of $\delta\mathrm{PWV}$
(including both intra-night and inter-night components).

## Expected behaviour

- At short lags (< 10 h) the intra-night decorrelation is visible as in PWV03.
- At multi-day lags the correlation should reflect synoptic weather patterns
  (e.g. 3–7 day weather cycles).
- At annual timescales a residual correlation from the seasonal cycle will
  appear unless the seasonal trend is subtracted first.  Both cases
  (with and without seasonal subtraction) are shown.

## Lag grid

Two grids are used:
- **Short**: 50 log-spaced bins from 1 min to 600 min (intra-night regime).
- **Long**: 60 log-spaced bins from 1 min to 500 days.

**Author :** Sylvie Dagoret-Campagne — IJCLab/IN2P3/CNRS  
**Created :** 2026-03-29  
**Kernel :** conda_py313

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import warnings
warnings.resetwarnings()
warnings.simplefilter('ignore')

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.lines import Line2D
from astropy.time import Time
from scipy.optimize import curve_fit

%matplotlib inline

mpl.rcParams.update({
    'figure.dpi'     : 150,
    'font.family'    : 'serif',
    'font.size'      : 11,
    'axes.labelsize' : 13,
    'axes.titlesize' : 12,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'legend.fontsize': 10,
    'axes.grid'      : True,
    'grid.alpha'     : 0.35,
    'axes.linewidth' : 1.1,
})

In [ ]:
from PWV00_parameters import (
    version_run, tag, extractedfilesdict,
    PWV_FILTER_LIST, FLAG_PWVFILTERS,
    DumpConfig
)
from mysitcom.auxtel.qualitycuts import ParameterCutSelection, ParameterCutTools
from mysitcom.auxtel.pwv import normalize_column_data_bytarget_byfilter

DumpConfig()

In [ ]:
# Output directory for long-scale correlation figures
pathfigs = 'figs_PWV03longscale'
prefix   = 'pwv03longscale'
os.makedirs(pathfigs, exist_ok=True)
figtype  = '.pdf'

In [ ]:
FLAG_WITHCOLLIMATOR     = True
datetime_WITHCOLLIMATOR = pd.to_datetime('2023-09-30 00:00:00.0+0000')

# Instrumental repeatability sigma (mm) — added in quadrature to variance
SIGMA_repeat = 0.12

---
## 1. Load data & build Time column

In [ ]:
# hack pour la lecture des pickles numpy
import numpy as np
import sys
sys.modules['numpy.rec'] = np.rec

In [ ]:
atmfilename   = extractedfilesdict[version_run]
inputfilename = atmfilename.split('/')[-1]
print(f'Loading: {atmfilename}')

if 'parquet' in inputfilename:
    df_spec = pd.read_parquet(atmfilename)
else:
    specdata = np.load(atmfilename, allow_pickle=True)
    df_spec  = pd.DataFrame(specdata)

df_spec['Time'] = pd.to_datetime(df_spec['DATE-OBS'], utc=True)
df_spec = df_spec.sort_values('Time', ascending=True).reset_index(drop=True)

if FLAG_WITHCOLLIMATOR:
    df_spec = df_spec[df_spec['Time'] > datetime_WITHCOLLIMATOR].copy()

print(f'Shape: {df_spec.shape}')
print(f'Date range: {df_spec["Time"].min().date()}  ->  {df_spec["Time"].max().date()}')

In [ ]:
df_spec['nightObs'] = df_spec.apply(lambda x: x['id'] // 100_000, axis=1)
df_spec['seq_num']  = df_spec['id'] % 100_000
df_spec['night_from_time_utc'] = df_spec['Time'].dt.strftime('%Y%m%d').astype(int)

FLAG_CORRECT_NIGHTOBS_VARIABLES = True

if FLAG_CORRECT_NIGHTOBS_VARIABLES and 'run2026_v02' in version_run:
    print('COMPUTE NIGHTOBS FROM Time')
    df_spec['Time_local'] = df_spec['Time'].dt.tz_convert('America/Santiago')
    df_spec['nightObs_corr'] = (
        (df_spec['Time_local'] - pd.Timedelta(hours=12))
        .dt.strftime('%Y%m%d')
        .astype(int)
    )
    df_spec['nightObs'] = df_spec['nightObs_corr']

In [ ]:
if 'run2026_v02' in version_run:
    df_spec['chi2_ram'] = df_spec['CHI2_FIT']
    df_spec['chi2_rum'] = df_spec['CHI2_FIT']

if FLAG_PWVFILTERS:
    df_spec = df_spec[df_spec['FILTER'].isin(PWV_FILTER_LIST)].copy()

pwv_cols = ['PWV [mm]_ram', 'PWV [mm]_rum', 'PWV [mm]_err_ram', 'PWV [mm]_err_rum']
df_spec.dropna(subset=pwv_cols, inplace=True)

print(f'After filter selection: {len(df_spec)} rows')

In [ ]:
df_spec, _ = normalize_column_data_bytarget_byfilter(
    df_spec, target_col='TARGET', filter_col='FILTER',
    feature_col='CHI2_FIT', ext='norm')
df_spec, _ = normalize_column_data_bytarget_byfilter(
    df_spec, target_col='TARGET', filter_col='FILTER',
    feature_col='chi2_ram', ext='norm')
df_spec, _ = normalize_column_data_bytarget_byfilter(
    df_spec, target_col='TARGET', filter_col='FILTER',
    feature_col='chi2_rum', ext='norm')

---
## 2. Quality cuts

In [ ]:
FLAG_LOOSE_CUTS = True
FLAG_TIGHT_CUTS = False

pathdata = 'data_PWV01seas'
if FLAG_LOOSE_CUTS:
    filename_cuts = f'{pathdata}/cuts_loose_finaldecision.json'
    cutstype_name = 'loose-cuts'
elif FLAG_TIGHT_CUTS:
    filename_cuts = f'{pathdata}/cuts_tight_finaldecision.json'
    cutstype_name = 'tight-cuts'
else:
    filename_cuts = f'{pathdata}/cuts_finaldecision.json'
    cutstype_name = 'standard-cuts'

cuts           = ParameterCutTools.load_cuts_json(filename_cuts)
list_of_params = list(cuts.keys())

selector    = ParameterCutSelection(df_spec, params=list_of_params, id_col='id')
flags       = selector.apply_cuts(cuts)
df_selected = df_spec.merge(flags, on='id')
df_keep     = df_selected[df_selected['pass_all_cuts']].copy()

print(f'{cutstype_name}: {len(df_keep)} / {len(df_spec)} kept')

---
## 3. Prepare PWV time series — global mean subtraction

### Why global mean subtraction (not per-night)?

In `PWV03` the nightly mean is subtracted so that only intra-night
fluctuations remain.  Here we want to keep the inter-night variability:
the slow modulation of PWV by synoptic weather and the seasonal cycle
is precisely the signal we aim to characterise at multi-day lags.

We therefore subtract only the **global mean** of the full dataset
(equivalent to the time-average over all observations).  The resulting
$\delta\mathrm{PWV}$ retains both intra-night and inter-night
fluctuations.  The normalisation variance $\sigma^2$ is accordingly
the total variance (intra + inter night combined).

A **seasonal-subtracted variant** is also provided (Section 3b): a
GP or sinusoidal seasonal model is subtracted to isolate residuals
on timescales shorter than ~1 year.

In [ ]:
# ── Best-estimate PWV ─────────────────────────────────────────────────────────
df_keep['PWV']     = df_keep['PWV [mm]_ram']
df_keep['PWV_err'] = df_keep['PWV [mm]_err_ram']

# Time in days since first observation (convenient for multi-day lags)
t0_mjd        = df_keep['MJD'].min()
df_keep['t_days'] = df_keep['MJD'] - t0_mjd
df_keep['t_min']  = df_keep['t_days'] * 1440.0

# ── Global mean subtraction ───────────────────────────────────────────────────
global_mean       = df_keep['PWV'].mean()
df_keep['dPWV']   = df_keep['PWV'] - global_mean

# Total variance (intra + inter night)
pwv_var_total = df_keep['dPWV'].var(ddof=1) + SIGMA_repeat**2

n_nights = df_keep['nightObs'].nunique()
print(f'N observations              : {len(df_keep)}')
print(f'N nights                    : {n_nights}')
print(f'Global mean PWV             : {global_mean:.3f} mm')
print(f'Total variance sigma^2      : {pwv_var_total:.4f} mm^2  => sigma = {np.sqrt(pwv_var_total):.3f} mm')
print(f'Time span                   : {df_keep["t_days"].max():.1f} days')

In [ ]:
# Quick overview: PWV time series
fig0, ax0 = plt.subplots(figsize=(16, 4))
ax0.scatter(df_keep['t_days'], df_keep['PWV'],
            s=2, alpha=0.4, color='steelblue', label='PWV (all obs)')
ax0.axhline(global_mean, color='darkred', lw=1.5, ls='--', label=f'Global mean = {global_mean:.2f} mm')
ax0.set_xlabel('Time [days since first obs]', fontsize=12)
ax0.set_ylabel('PWV [mm]', fontsize=12)
ax0.set_title(f'PWV time series — {version_run}', fontsize=11)
ax0.legend(fontsize=9)
fig0.tight_layout()
fig0.savefig(f'{pathfigs}/{prefix}_{version_run}_timeseries.pdf', bbox_inches='tight')
plt.show()

### 3b. Optional: seasonal subtraction

If a smooth seasonal model is available (e.g. from PWV02 GP fit), it can be
subtracted here to isolate residuals on timescales shorter than ~1 year.
For now we provide a simple sinusoidal fit as a placeholder.

In [ ]:
FLAG_SUBTRACT_SEASONAL = True   # set True to activate

if FLAG_SUBTRACT_SEASONAL:
    from scipy.optimize import curve_fit as _cf

    def _seasonal_model(t_days, A, phi, offset):
        """Simple annual sinusoidal model."""
        return A * np.sin(2 * np.pi * t_days / 365.25 + phi) + offset

    popt_seas, _ = _cf(
        _seasonal_model,
        df_keep['t_days'].values,
        df_keep['PWV'].values,
        p0=[1.0, 0.0, global_mean],
        maxfev=10_000
    )
    seasonal_fit        = _seasonal_model(df_keep['t_days'].values, *popt_seas)
    df_keep['dPWV_seas'] = df_keep['PWV'] - seasonal_fit
    pwv_var_seas         = df_keep['dPWV_seas'].var(ddof=1) + SIGMA_repeat**2
    print(f'Seasonal-subtracted sigma : {np.sqrt(pwv_var_seas):.3f} mm')
else:
    df_keep['dPWV_seas'] = df_keep['dPWV']
    pwv_var_seas         = pwv_var_total
    print('No seasonal subtraction applied (FLAG_SUBTRACT_SEASONAL = False)')

---
## 4. Two-point correlation — all-pairs estimator on long lag grid

All ordered pairs $(i, j)$ with $j > i$ (across the entire dataset,
including cross-night pairs) are formed.  The lag
$\Delta t_{ij} = t_j - t_i$ is expressed in **minutes** and binned on
a **logarithmic grid** from 1 min to ~500 days.

For datasets with thousands of observations the number of pairs grows
as $N(N-1)/2$, which can reach $10^7$–$10^8$ for $N \sim 10^4$.
We therefore implement an optional **sub-sampling** per time-segment
to keep the computation tractable.

In [ ]:
# ── Parameters ────────────────────────────────────────────────────────────────
LAG_MIN_MIN  =    1.0          # 1 minute
#LAG_MAX_DAY  =  500.0          # 500 days
LAG_MAX_DAY  =  180.0          # 50 days
LAG_MAX_MIN  = LAG_MAX_DAY * 1440.0

N_BINS_LOG   =  60             # log-spaced bins
MIN_PAIRS    =   10             # minimum pairs per bin for display

# Sub-sampling: if the dataset has more than N_MAX points, draw a random
# subset to limit the number of pairs.
N_MAX_SUBSAMPLE = 5000
rng = np.random.default_rng(seed=42)

print(f'Lag range: {LAG_MIN_MIN:.0f} min — {LAG_MAX_DAY:.0f} days ({LAG_MAX_MIN:.0f} min)')
print(f'N observations in df_keep: {len(df_keep)}')

In [ ]:
PLOT_XSCALE = "log"
PLOT_YMIN = -0.5
PLOT_YMAX = 1.2

In [ ]:
# ── Bin edges (log-spaced in minutes) ─────────────────────────────────────────
bin_edges       = np.logspace(np.log10(LAG_MIN_MIN), np.log10(LAG_MAX_MIN), N_BINS_LOG + 1)
bin_centers_min = np.sqrt(bin_edges[:-1] * bin_edges[1:])   # geometric mean
bin_centers_day = bin_centers_min / 1440.0
bin_centers_hr  = bin_centers_min / 60.0

In [ ]:
# ── Form all pairs (with optional sub-sampling) ───────────────────────────────
# We use the global-mean-subtracted residuals.

t_all   = df_keep['t_min'].values
dp_all  = df_keep['dPWV'].values
N_all   = len(t_all)

if N_all > N_MAX_SUBSAMPLE:
    print(f'Sub-sampling {N_MAX_SUBSAMPLE} / {N_all} observations...')
    idx_sub = np.sort(rng.choice(N_all, N_MAX_SUBSAMPLE, replace=False))
    t_use   = t_all[idx_sub]
    dp_use  = dp_all[idx_sub]
else:
    t_use   = t_all
    dp_use  = dp_all

print(f'Using {len(t_use)} observations -> {len(t_use)*(len(t_use)-1)//2:,} pairs (before lag cut)')

# Upper-triangle pairs
ii, jj   = np.triu_indices(len(t_use), k=1)
dt_pairs = t_use[jj]  - t_use[ii]     # minutes, >= 0
pr_pairs = dp_use[ii] * dp_use[jj]

# Keep only pairs within the lag window
ok = (dt_pairs >= LAG_MIN_MIN) & (dt_pairs <= LAG_MAX_MIN)
dt_pairs = dt_pairs[ok]
pr_pairs = pr_pairs[ok]

print(f'Pairs within [{LAG_MIN_MIN:.0f} min, {LAG_MAX_DAY:.0f} days] : {len(dt_pairs):,}')

In [ ]:
# ── Bin the pairs ─────────────────────────────────────────────────────────────
bin_idx = np.digitize(dt_pairs, bin_edges) - 1
valid   = (bin_idx >= 0) & (bin_idx < N_BINS_LOG)

corr     = np.full(N_BINS_LOG, np.nan)
corr_err = np.full(N_BINS_LOG, np.nan)
n_pairs  = np.zeros(N_BINS_LOG, dtype=int)

for k in range(N_BINS_LOG):
    mask = valid & (bin_idx == k)
    nk   = mask.sum()
    n_pairs[k] = nk
    if nk == 0:
        continue
    pk          = pr_pairs[mask]
    corr[k]     = pk.mean() / pwv_var_total
    corr_err[k] = pk.std(ddof=1) / (np.sqrt(nk) * pwv_var_total) if nk > 1 else np.nan

good = n_pairs >= MIN_PAIRS
print(f'Bins with >= {MIN_PAIRS} pairs: {good.sum()} / {N_BINS_LOG}')

---
## 5. Main figure — full lag range (log days)

Left panel: full range 1 min – 500 days (log scale in days).
Right panel: zoom on intra-night regime (log scale in minutes, 1–600 min).

In [ ]:
CORR_COLOR  = 'steelblue'
ZERO_COLOR  = 'gray'
INV_E_COLOR = 'darkorange'
ONE_DAY_COLOR = 'purple'
WEEK_COLOR    = 'green'

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
fig.subplots_adjust(wspace=0.32)

# ── LEFT : full range, log x in days ──────────────────────────────────────────
ax_left = axes[0]
ax_left.fill_between(
    bin_centers_day[good],
    corr[good] - corr_err[good],
    corr[good] + corr_err[good],
    alpha=0.25, color=CORR_COLOR
)
ax_left.plot(bin_centers_day[good], corr[good],
             color=CORR_COLOR, lw=1.8, marker='o', ms=3, zorder=3,
             label=r'$C(\Delta t)$')
ax_left.axhline(0.0,        ls='--', color=ZERO_COLOR,    lw=1.0)
ax_left.axhline(1.0/np.e,   ls=':',  color=INV_E_COLOR,   lw=1.5, label=r'$C=1/e$')
ax_left.axvline(1.0,        ls='-',  color=ONE_DAY_COLOR, lw=1.2, alpha=0.6, label='1 day')
ax_left.axvline(7.0,        ls='-',  color=WEEK_COLOR,    lw=1.2, alpha=0.6, label='1 week')
ax_left.axvline(30.0,       ls='--', color=WEEK_COLOR,    lw=1.0, alpha=0.5, label='1 month')
ax_left.axvline(365.25,     ls='--', color='red',          lw=1.0, alpha=0.5, label='1 year')
ax_left.set_xscale(PLOT_XSCALE)
ax_left.set_xlim(LAG_MIN_MIN / 1440.0, LAG_MAX_DAY)
ax_left.set_ylim(PLOT_YMIN, PLOT_YMAX)
ax_left.set_xlabel(r'$\Delta t$ [days]', fontsize=13)
ax_left.set_ylabel(r'$C(\Delta t)$', fontsize=13)
ax_left.set_title('Full lag range — log days (1 min – 500 d)', fontsize=11)
ax_left.legend(fontsize=9, loc='upper right')

# secondary x-axis: minutes at bottom, hours at top
ax_left2 = ax_left.twiny()
ax_left2.set_xscale(PLOT_XSCALE)
ax_left2.set_xlim(np.array(ax_left.get_xlim()) * 1440.0)
ax_left2.set_xlabel(r'$\Delta t$ [minutes]', fontsize=10, labelpad=6)

# ── RIGHT : intra-night zoom (1–600 min, log) ──────────────────────────────────
ax_right = axes[1]
zoom_max_min = 600.0
zoom_mask = good & (bin_centers_min <= zoom_max_min)

ax_right.fill_between(
    bin_centers_min[zoom_mask],
    corr[zoom_mask] - corr_err[zoom_mask],
    corr[zoom_mask] + corr_err[zoom_mask],
    alpha=0.25, color=CORR_COLOR
)
ax_right.plot(bin_centers_min[zoom_mask], corr[zoom_mask],
              color=CORR_COLOR, lw=1.8, marker='o', ms=4, zorder=3,
              label=r'$C(\Delta t)$')
ax_right.axhline(0.0,      ls='--', color=ZERO_COLOR,  lw=1.0)
ax_right.axhline(1.0/np.e, ls=':',  color=INV_E_COLOR, lw=1.5, label=r'$C=1/e$')
ax_right.set_xscale(PLOT_XSCALE)
ax_right.set_xlim(LAG_MIN_MIN, zoom_max_min)
ax_right.set_ylim(PLOT_YMIN, PLOT_YMAX)
ax_right.set_xlabel(r'$\Delta t$ [minutes]', fontsize=13)
ax_right.set_ylabel(r'$C(\Delta t)$', fontsize=13)
ax_right.set_title('Intra-night zoom (1–600 min)', fontsize=11)
ax_right.legend(fontsize=9, loc='upper right')

fig.suptitle(
    f'PWV two-point correlation — global mean subtracted, all pairs\n'
    f'{version_run}',
    fontsize=12, y=1.02
)
figfile1 = f'{pathfigs}/{prefix}_{version_run}_2point_corr_longscale.pdf'
fig.savefig(figfile1, bbox_inches='tight')
print(f'Saved: {figfile1}')
plt.show()

---
## 6. Focus on multi-day regime (1 day – 100 days)

In [ ]:
fig2, ax2 = plt.subplots(figsize=(12, 5))

day_mask = good & (bin_centers_day >= 0.5) & (bin_centers_day <= 100.0)

ax2.fill_between(
    bin_centers_day[day_mask],
    corr[day_mask] - corr_err[day_mask],
    corr[day_mask] + corr_err[day_mask],
    alpha=0.3, color=CORR_COLOR
)
ax2.plot(bin_centers_day[day_mask], corr[day_mask],
         color=CORR_COLOR, lw=2, marker='o', ms=5, zorder=3,
         label=r'$C(\Delta t)$')
ax2.axhline(0.0,      ls='--', color=ZERO_COLOR,  lw=1.0)
ax2.axhline(1.0/np.e, ls=':',  color=INV_E_COLOR, lw=1.5, label=r'$C=1/e$')
for day_mark, label_mark, col_mark in [
    (1.0, '1 day', ONE_DAY_COLOR),
    (7.0, '7 days', WEEK_COLOR),
    (30.0, '30 days', 'olive'),
]:
    ax2.axvline(day_mark, ls='--', color=col_mark, lw=1.2, alpha=0.7, label=label_mark)

ax2.set_xscale(PLOT_XSCALE)
ax2.set_xlim(0.5, 100.0)
ax2.set_ylim(PLOT_YMIN, PLOT_YMAX)
ax2.set_xlabel(r'$\Delta t$ [days]', fontsize=13)
ax2.set_ylabel(r'$C(\Delta t)$', fontsize=13)
ax2.set_title(r'Multi-day PWV correlation (0.5 – 100 days, log scale)', fontsize=11)
ax2.legend(fontsize=9, loc='upper right')

fig2.suptitle(f'PWV long-scale correlation — {version_run}', fontsize=12)
figfile2 = f'{pathfigs}/{prefix}_{version_run}_multiday.pdf'
fig2.savefig(figfile2, bbox_inches='tight')
print(f'Saved: {figfile2}')
plt.show()

---
## 7. Pair count per bin

In [ ]:
fig3, ax3 = plt.subplots(figsize=(14, 4))
ax3.bar(np.arange(N_BINS_LOG)[good], n_pairs[good],
        width=0.8, color='steelblue', alpha=0.8, edgecolor='white', linewidth=0.4)
ax3.set_yscale('log')
ax3.set_xlabel('Bin index', fontsize=12)
ax3.set_ylabel('Number of pairs', fontsize=12)
ax3.set_title(f'Pair count per log-lag bin ({LAG_MIN_MIN:.0f} min – {LAG_MAX_DAY:.0f} days)', fontsize=11)

tick_idx = np.linspace(0, N_BINS_LOG - 1, 10, dtype=int)
ax3.set_xticks(tick_idx)
ax3.set_xticklabels(
    [f'{bin_centers_min[k]:.0f} min' if bin_centers_day[k] < 1.0
     else f'{bin_centers_day[k]:.1f} d'
     for k in tick_idx],
    rotation=35, ha='right', fontsize=9
)
fig3.tight_layout()
figfile3 = f'{pathfigs}/{prefix}_{version_run}_pair_counts.pdf'
fig3.savefig(figfile3, bbox_inches='tight')
print(f'Saved: {figfile3}')
plt.show()

---
## 8. Per-filter correlation on the long lag grid

In [ ]:
FILTERS_OF_INTEREST = ['empty', 'OG550_65mm_1', 'FELH0600']
filter_color = {
    'empty'        : '#1f77b4',
    'OG550_65mm_1' : '#d62728',
    'FELH0600'     : '#2ca02c',
}
filter_label = {
    'empty'        : 'empty',
    'OG550_65mm_1' : 'OG550',
    'FELH0600'     : 'FELH0600',
}

corr_by_filter     = {}
corr_err_by_filter = {}
npairs_by_filter   = {}

for filt in FILTERS_OF_INTEREST:
    sub_f = df_keep[df_keep['FILTER'] == filt].copy()
    if len(sub_f) < 10:
        print(f'  {filt}: too few points ({len(sub_f)}), skipped')
        continue

    # Global mean subtraction per filter
    gm_f             = sub_f['PWV'].mean()
    sub_f['dPWV_f']  = sub_f['PWV'] - gm_f
    var_f            = sub_f['dPWV_f'].var(ddof=1) + SIGMA_repeat**2

    t_f   = sub_f['t_min'].values
    dp_f  = sub_f['dPWV_f'].values
    N_f   = len(t_f)

    if N_f > N_MAX_SUBSAMPLE:
        idx_f = np.sort(rng.choice(N_f, N_MAX_SUBSAMPLE, replace=False))
        t_f   = t_f[idx_f]
        dp_f  = dp_f[idx_f]

    ii_f, jj_f = np.triu_indices(len(t_f), k=1)
    dt_f       = t_f[jj_f]  - t_f[ii_f]
    pr_f       = dp_f[ii_f] * dp_f[jj_f]
    ok_f       = (dt_f >= LAG_MIN_MIN) & (dt_f <= LAG_MAX_MIN)
    dt_f, pr_f = dt_f[ok_f], pr_f[ok_f]

    bidx_f = np.digitize(dt_f, bin_edges) - 1
    val_f  = (bidx_f >= 0) & (bidx_f < N_BINS_LOG)

    c_f   = np.full(N_BINS_LOG, np.nan)
    ce_f  = np.full(N_BINS_LOG, np.nan)
    np_f  = np.zeros(N_BINS_LOG, dtype=int)

    for k in range(N_BINS_LOG):
        m  = val_f & (bidx_f == k)
        nk = m.sum()
        np_f[k] = nk
        if nk < MIN_PAIRS:
            continue
        pk     = pr_f[m]
        c_f[k]  = pk.mean() / var_f
        ce_f[k] = pk.std(ddof=1) / (np.sqrt(nk) * var_f) if nk > 1 else np.nan

    corr_by_filter[filt]     = c_f
    corr_err_by_filter[filt] = ce_f
    npairs_by_filter[filt]   = np_f
    print(f'  {filt}: {len(sub_f)} obs, {len(dt_f):,} pairs (after lag cut)')

print('Done.')

In [ ]:
fig4, axes4 = plt.subplots(1, 2, figsize=(18, 6))
fig4.subplots_adjust(wspace=0.32)

for ax4, xdata, xlabel, xscale, xlim in [
    (axes4[0], bin_centers_day, r'$\Delta t$ [days]',    'log', (LAG_MIN_MIN/1440.0, LAG_MAX_DAY)),
    (axes4[1], bin_centers_day, r'$\Delta t$ [days]',    'log', (0.5, 100.0)),
]:
    ax4.plot(bin_centers_day[good], corr[good],
             color='black', lw=1.0, alpha=0.4, ls='--', label='All filters')
    for filt in FILTERS_OF_INTEREST:
        if filt not in corr_by_filter:
            continue
        c_f  = corr_by_filter[filt]
        ce_f = corr_err_by_filter[filt]
        gf   = npairs_by_filter[filt] >= MIN_PAIRS
        ax4.fill_between(xdata[gf], c_f[gf] - ce_f[gf], c_f[gf] + ce_f[gf],
                         alpha=0.15, color=filter_color[filt])
        ax4.plot(xdata[gf], c_f[gf],
                 color=filter_color[filt], lw=1.8, marker='o', ms=3,
                 label=filter_label[filt])
    ax4.axhline(0.0,        ls='--', color=ZERO_COLOR,    lw=1.0)
    ax4.axhline(1.0/np.e,   ls=':',  color=INV_E_COLOR,   lw=1.5, label=r'$C=1/e$')
    ax4.axvline(1.0,        ls='-',  color=ONE_DAY_COLOR, lw=1.2, alpha=0.5, label='1 day')
    ax4.axvline(7.0,        ls='-',  color=WEEK_COLOR,    lw=1.2, alpha=0.5, label='7 days')
    ax4.set_xscale(PLOT_XSCALE)
    ax4.set_xlim(xlim)
    ax4.set_ylim(PLOT_YMIN, PLOT_YMAX)
    ax4.set_xlabel(xlabel, fontsize=13)
    ax4.set_ylabel(r'$C(\Delta t)$', fontsize=13)
    ax4.legend(fontsize=9, loc='upper right')

axes4[0].set_title('Per-filter long-scale correlation', fontsize=11)
axes4[1].set_title('Per-filter — zoom 0.5–100 days', fontsize=11)
fig4.suptitle(f'PWV long-scale correlation by filter — {version_run}', fontsize=12, y=1.01)
figfile4 = f'{pathfigs}/{prefix}_{version_run}_per_filter_longscale.pdf'
fig4.savefig(figfile4, bbox_inches='tight')
print(f'Saved: {figfile4}')
plt.show()

---
## 9. Summary & export

In [ ]:
df_corr_out = pd.DataFrame({
    'lag_min' : bin_centers_min,
    'lag_hr'  : bin_centers_hr,
    'lag_day' : bin_centers_day,
    'C'       : corr,
    'C_err'   : corr_err,
    'n_pairs' : n_pairs,
})
csv_file = f'{pathfigs}/{prefix}_{version_run}_2point_corr_longscale.csv'
df_corr_out.to_csv(csv_file, index=False, float_format='%.6f')
print(f'Saved: {csv_file}')

print(f'\n=== PWV long-scale two-point correlation summary ===')
print(f'  N observations (after cuts)  : {len(df_keep)}')
print(f'  N nights                     : {n_nights}')
print(f'  Global mean PWV              : {global_mean:.3f} mm')
print(f'  Total sigma                  : {np.sqrt(pwv_var_total):.3f} mm')
print(f'  Total pairs (after lag cut)  : {len(dt_pairs):,}')
print(f'  Lag range                    : {LAG_MIN_MIN:.0f} min – {LAG_MAX_DAY:.0f} days')
print(f'  N log bins                   : {N_BINS_LOG}')
print(f'  Bins with >= {MIN_PAIRS} pairs: {good.sum()} / {N_BINS_LOG}')